In [10]:
!pip install transformers torch

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load DialoGPT model & tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

print("Model loaded successfully!")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [ ]:
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant . Type 'exit' or 'quit' to stop.\n")

while True:
    # Take user input
    user_input = input("You: ")

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! ")
        break

    # Encode user input with EOS token
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append input to chat history for context
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Create attention mask (fixes warning)
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    # Generate response using sampling (better quality)
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        attention_mask=attention_mask,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    # Limit chat history (improves speed & memory)
    chat_history_ids = chat_history_ids[:, -1000:]

    # Decode only the bot response
    bot_response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    # Handle empty or weird responses
    if bot_response.strip() == "":
        bot_response = "I'm not sure about that. Can you ask something else?"

    # Print response
    print("Chatbot:", bot_response)

Chatbot: Hello! I am your AI assistant . Type 'exit' or 'quit' to stop.

